# Cookbook 03: Regime-Aware Trading

Use fractime's HurstRegimeAnalyzer to detect market regimes, then adapt trading strategy allocation.

```bash
pip install wrdata fractime wrtrade
```

In [ ]:
from wrdata import DataStream
import fractime as ft
import wrtrade as wrt
import polars as pl
import numpy as np

## Step 1: Fetch SPY Data

In [ ]:
stream = DataStream()
df = stream.get("SPY", start="2023-01-01", interval="1d")
prices_np = df["close"].to_numpy()
prices_pl = df["close"]
print(f"Fetched {len(prices_np)} days of SPY data")

## Step 2: Detect Regime

In [ ]:
regime_analyzer = ft.HurstRegimeAnalyzer()
regime_result = regime_analyzer.analyze(prices_np)

print(f"Current Hurst: {regime_result.hurst:.3f}")
print(f"Regime: {regime_result.regime}")
print(f"Confidence: {regime_result.confidence:.2%}")

## Step 3: Build Regime-Adaptive Portfolio

In [ ]:
def momentum_signal(prices: pl.Series) -> pl.Series:
    ret = prices.pct_change(20)
    return (ret > 0.02).cast(int) - (ret < -0.02).cast(int)

def reversion_signal(prices: pl.Series) -> pl.Series:
    sma = prices.rolling_mean(20)
    std = prices.rolling_std(20)
    z = (prices - sma) / std
    return (z < -1.5).cast(int) - (z > 1.5).cast(int)

# Weight based on regime
if regime_result.hurst > 0.55:
    mom_weight, rev_weight = 0.8, 0.2
    print("Trending regime - favoring momentum")
elif regime_result.hurst < 0.45:
    mom_weight, rev_weight = 0.2, 0.8
    print("Mean-reverting regime - favoring reversion")
else:
    mom_weight, rev_weight = 0.5, 0.5
    print("Random walk - equal weights")

builder = wrt.NDimensionalPortfolioBuilder()
portfolio = builder.create_portfolio("Regime_Adaptive", [
    builder.create_signal_component("Momentum", momentum_signal, weight=mom_weight),
    builder.create_signal_component("Reversion", reversion_signal, weight=rev_weight),
])

## Step 4: Backtest & Validate

In [ ]:
manager = wrt.AdvancedPortfolioManager(portfolio)
results = manager.backtest(prices_pl)

print(f"Total Return: {results['total_return']:.2%}")
print(f"Sortino Ratio: {results['portfolio_metrics']['sortino_ratio']:.2f}")
print(f"Max Drawdown: {results['portfolio_metrics']['max_drawdown']:.2%}")

test = manager.run_permutation_test(prices_pl, n_permutations=500, metric="sortino_ratio")
print(f"\nPermutation Test P-Value: {test['p_value']:.4f}")

## Step 5: Forecast

In [ ]:
forecaster = ft.Forecaster(n_paths=1000)
forecaster.fit(prices_np)
forecast = forecaster.predict(horizon=30)

print(f"Current: ${prices_np[-1]:.2f}")
print(f"Forecast: ${forecast.mean[-1]:.2f}")
print(f"Range: ${forecast.lower[-1]:.2f} - ${forecast.upper[-1]:.2f}")

fig = ft.plot_forecast(prices_np[-100:], forecast, title="SPY Regime-Informed Forecast")
fig.show()